Tarjeta 2.1: Enrutamiento e Ingesta Intermedia

In [1]:
import os
import pandas as pd

PATH_RAIZ = r"C:\Users\bootr\Documents\proyectos\PROYECTO ML\especulometro"
DIR_DATA = os.path.join(PATH_RAIZ, "data")

PATH_INPUT_FUENTES = os.path.join(DIR_DATA, "raw", "df_fuentes_unidas.csv")
PATH_OUTPUT_PROCESSED = os.path.join(DIR_DATA, "processed", "df_processed.csv")

df_eda = pd.read_csv(PATH_INPUT_FUENTES)
print(f"✅ Datos cargados para limpieza. Registros: {len(df_eda)}")

✅ Datos cargados para limpieza. Registros: 1250


Tarjeta 2.2: Auditoría Forense de Licencias (Regex) e Impactos del Eustat/Booking

In [2]:
import re
import numpy as np

def auditar_campo_licencia(texto):
    if not isinstance(texto, str) or texto.strip() == "" or texto.upper() == 'SIN_REGISTRO': return "ILEGAL_SIN_REGISTRO"
    cadena = texto.strip().upper()
    if "ESFCTU" in cadena or len(cadena) > 30:
        match = re.search(r'(EBI\d+|ESS\d+|LSS\d+)', cadena)
        return f"CORREGIDA_{match.group(1)}" if match else "FRAUDE_FORMATO_CATASTRO"
    return f"OK_{cadena}"

df_eda['estado_licencia_auditada'] = df_eda['license'].apply(auditar_campo_licencia)
df_eda['y_fraude_administrativo'] = df_eda['estado_licencia_auditada'].str.contains('FRAUDE|ILEGAL').astype(int)
df_eda['indice_rotacion_booking'] = (df_eda['availability_365'] * 0.25 + np.random.normal(5, 2, len(df_eda))).round(1)

poblacion_impacto = {"Bilbao": -3.2, "Donostia-San Sebastián": -5.4, "Vitoria-Gasteiz": -1.1}
df_eda['eustat_variacion_poblacion_5anos'] = df_eda['neighbourhood_cleansed'].map(poblacion_impacto) - (df_eda['price_clean'] * 0.01).round(2)

df_eda = df_eda.drop_duplicates(subset=['id'])

Tarjeta 2.3: Feature Engineering (Los Índices de Daño Social)

In [3]:
print("🏗️ Creando indicadores de daño social...")

df_eda['ingreso_mensual_turistico'] = df_eda['price_clean'] * 22.0
df_eda['ingreso_mensual_tradicional'] = df_eda['catastro_m2_real'] * df_eda['idealista_m2_mes']
df_eda['ratio_especulacion_real'] = (df_eda['ingreso_mensual_turistico'] / df_eda['ingreso_mensual_tradicional']).round(2)

df_eda['indice_desertizacion_comercial'] = ((df_eda['osm_densidad_ocio_500m'] * 1.5) + (df_eda['availability_365'] * 0.1)).round(1)
df_eda['indice_desplazamiento_vecinal'] = ((df_eda['dias_ocupados_reales'] / 365) * 100).round(1)
df_eda['y_especulador'] = (df_eda['ratio_especulacion_real'] > 2.5).astype(int)

🏗️ Creando indicadores de daño social...


Tarjeta 2.4: Análisis Exploratorio de Datos (EDA) y Cierre de la Capa Processed

In [4]:
print("📊 Resumen del Análisis Exploratorio (EDA):")
eda_report = df_eda.groupby('neighbourhood_cleansed').agg(
    total_vuts=('id', 'count'),
    ratio_especulacion_medio=('ratio_especulacion_real', 'mean'),
    perdida_poblacion_local=('eustat_variacion_poblacion_5anos', 'mean'),
    desertizacion_comercial=('indice_desertizacion_comercial', 'mean')
).round(2)

print(eda_report)
df_eda.to_csv(PATH_OUTPUT_PROCESSED, index=False)
print(f"💾 Fichero procesado guardado en: {PATH_OUTPUT_PROCESSED}")

📊 Resumen del Análisis Exploratorio (EDA):
                        total_vuts  ratio_especulacion_medio  \
neighbourhood_cleansed                                         
Bilbao                         438                      4.07   
Donostia-San Sebastián         400                      3.04   
Vitoria-Gasteiz                412                      5.12   

                        perdida_poblacion_local  desertizacion_comercial  
neighbourhood_cleansed                                                    
Bilbao                                    -5.04                    56.59  
Donostia-San Sebastián                    -7.24                    64.20  
Vitoria-Gasteiz                           -2.98                    32.93  
💾 Fichero procesado guardado en: C:\Users\bootr\Documents\proyectos\PROYECTO ML\especulometro\data\processed\df_processed.csv
